In [4]:
import pandas as pd
from pyecharts.charts import Bar
from pyecharts import options as opts
from pyecharts.globals import ThemeType
from IPython.display import HTML, display

# ------------------------------
# 1. 加载和处理数据
# ------------------------------

try:
    df_orders = pd.read_csv('../../erp_order.csv')
except FileNotFoundError:
    print(f"错误: erp_order.csv 未找到。")
    print("请先运行您提供的 Python 脚本来生成数据文件。")
    exit()

platforms = ["淘宝", "天猫", "京东", "抖音", "拼多多", "微信"]
platform_counts = df_orders.groupby('平台站点')['内部订单号'].count()
platform_counts = platform_counts.reindex(platforms, fill_value=0).sort_values(ascending=False)

x_data = platform_counts.index.tolist()
y_data = [int(val) for val in platform_counts.values.tolist()]

# ------------------------------
# 2. 定义渐变色 (使用 ECharts 字典格式)
# ------------------------------
gradient_color = {
    "type": "linear",  # 线性渐变
    "x": 0,            # 渐变起始点 x 坐标 (0 为最左)
    "y": 0,            # 渐变起始点 y 坐标 (0 为最上)
    "x2": 0,           # 渐变结束点 x 坐标
    "y2": 1,           # 渐变结束点 y 坐标 (1 为最下)
    "colorStops": [
        {"offset": 0, "color": "#29E8FF"},
        {"offset": 1, "color": "#00A1FF"}
    ],
    "global": False
}

# ------------------------------
# 3. 绘制图表
# ------------------------------

bar_chart = Bar(
    init_opts=opts.InitOpts(
        theme=ThemeType.DARK,
        bg_color="#0a192f",
        width="900px",
        height="500px"
    )
)

bar_chart.add_xaxis(xaxis_data=x_data)

bar_chart.add_yaxis(
    series_name="订单数量",
    y_axis=y_data,
    
    # --- 关键样式 ---
    itemstyle_opts=opts.ItemStyleOpts(
        color=gradient_color,      # type: ignore
        border_radius=[20, 20, 20, 20], # type: ignore
    ),
    
    # --- 顶部标签 ---
    label_opts=opts.LabelOpts(
        is_show=True,
        position="top",
        color="#FFFFFF",
        font_size=14,
        font_weight="bold",
        background_color="#202A4D",
        padding=[4, 8, 4, 8],
        border_radius=4,
    ),
    
    bar_width="40%",
)

# 4. 设置全局选项
bar_chart.set_global_opts(
    yaxis_opts=opts.AxisOpts(
        max_=max(y_data) + 100 if y_data else 1000,  # 动态设置最大值
        is_show=True,
        axislabel_opts=opts.LabelOpts(color="#FFFFFF", font_size=12),
        axistick_opts=opts.AxisTickOpts(is_show=False),
        axisline_opts=opts.AxisLineOpts(is_show=False),
        splitline_opts=opts.SplitLineOpts(
            is_show=True,
            linestyle_opts=opts.LineStyleOpts(
                type_="dashed",
                color="#FFFFFF",
                opacity=0.2
            )
        )
    ),
    xaxis_opts=opts.AxisOpts(
        axislabel_opts=opts.LabelOpts(color="#FFFFFF", font_size=12),
        axistick_opts=opts.AxisTickOpts(is_show=False),
        axisline_opts=opts.AxisLineOpts(is_show=False),
    ),
    legend_opts=opts.LegendOpts(is_show=False),
    title_opts=opts.TitleOpts(
        is_show=True,  # 显示标题
        title="电商平台订单量分布",  # 标题文本
        pos_left="center",  # 标题位置居中
        title_textstyle_opts=opts.TextStyleOpts(
            color="#FFFFFF",  # 标题颜色
            font_size=18,  # 标题字体大小
            font_weight="bold"  # 标题字体加粗
        )
    ),
    tooltip_opts=opts.TooltipOpts(
        trigger="axis", 
        axis_pointer_type="shadow"
    )
)

# 5. 在Jupyter Notebook中显示图表
chart_html = bar_chart.render_embed()  # 使用render_embed()来嵌入图表
display(HTML(chart_html))